# Laboratorio 10b — Entrenar bien la red
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 11 de noviembre de 2026 · Clase 10 (día doble)
**Bloque:** Lab 2 · 12:45–13:50 (65 min), después de la Teoría 2
**Viene de:** Teoría 2 (valle estrecho, momentum, RMSprop y Adam, sobreajuste, L2 y L1, Dropout, Batch Normalization, receta práctica) y del Lab 10a, donde se armó la red de dígitos y la función `entrenar`.

El notebook es **autocontenido**: la primera celda recrea los datos, las particiones, la red base y `entrenar`, así que se puede reiniciar el kernel y ejecutar desde arriba.

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **A** | Optimizadores a mano en el valle de las láminas | 15 min |
| **B** | Optimizadores en la red de dígitos | 15 min |
| **C** | Provocar sobreajuste | 8 min |
| **D** | Remedios, uno por uno (L2, L1, BatchNorm, Dropout, early stopping) | 20 min |
| **E** | Informe con incertidumbre | 7 min |

## Objetivos de aprendizaje

1. Ver por qué un η fijo zigzaguea en un valle estrecho y cómo lo corrigen momentum, RMSprop y Adam.
2. Comparar optimizadores en una red real, con la misma inicialización y el mismo orden de mini-batches.
3. Provocar, diagnosticar y combatir el sobreajuste con **un remedio a la vez**.
4. Reportar el resultado final con incertidumbre (varias semillas e IC95 % por bootstrap), mirando la prueba **una sola vez**.

> **Cómo se trabaja.** Las celdas con `# TODO` se completan; las celdas «Autocomprobación» fallan con un mensaje si algo no está bien. Las preguntas se responden en la celda de texto que sigue a cada una.

In [ ]:
import sys, time, copy
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy scikit-learn matplotlib pandas torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

SEED = 42
RAPIDO = False   # True: menos épocas, solo para probar el notebook rápido (valores del libreto = False)
np.random.seed(SEED); torch.manual_seed(SEED)
T0 = time.time()

EPOCAS = {"opt": 30, "sobreajuste": 200}
if RAPIDO:
    EPOCAS = {"opt": 10, "sobreajuste": 60}

AZUL, NARANJA, MARINO, GRIS = "#2e69b2", "#ba6826", "#1b2740", "#8b929c"   # colores de las láminas
plt.rcParams.update({
    "figure.facecolor": "#f2f1ee", "axes.facecolor": "#fffefb", "axes.edgecolor": "#dcdad3",
    "axes.grid": True, "grid.color": "#e3e1da", "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS, "#9aa6b8"]),
    "axes.titlesize": 12, "axes.labelsize": 11, "figure.dpi": 100,
})

# --- Datos y particiones: entrenamiento 1.047 | validación 300 | prueba 450 (estratificadas) ---
digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.25, stratify=y_all, random_state=SEED)
X_pool, X_val, y_pool, y_val = train_test_split(X_tr, y_tr, test_size=300, stratify=y_tr, random_state=SEED)

def a_tensores(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

Xp_t, yp_t = a_tensores(X_pool, y_pool)
Xv_t, yv_t = a_tensores(X_val, y_val)
Xte_t, yte_t = a_tensores(X_te, y_te)   # la prueba NO se toca hasta la Parte E

# --- Red base 64-64-32-10 y función `entrenar` del Lab 10a ---
def crear_red_base():
    """Red 64 -> 64 -> 32 -> 10 con ReLU."""
    return nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))


def evaluar(modelo, X, y):
    """Pérdida, exactitud y predicciones del modelo en modo evaluación (sin gradientes)."""
    modelo.eval()
    with torch.no_grad():
        logits = modelo(X)
        perd = nn.functional.cross_entropy(logits, y).item()
        pred = logits.argmax(dim=1)
    return {"loss": perd, "acc": (pred == y).float().mean().item(), "pred": pred.numpy()}


def entrenar_una_epoca(modelo, optimizador, loader, criterio, penal=None):
    """Una pasada por el DataLoader. Devuelve la pérdida media de la época.
    `penal` (opcional, se usa desde la Parte D): función modelo -> término que se suma a la pérdida."""
    modelo.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        optimizador.zero_grad()                       # 1 · 2
        loss = criterio(modelo(xb), yb)
        if penal is not None:
            loss = loss + penal(modelo)
        loss.backward()                               # 3
        optimizador.step()                            # 4
        total += loss.item() * len(xb)
        n += len(xb)
    return total / n


def entrenar(modelo, optimizador, X_tr, y_tr, X_ev, y_ev, epocas=30, batch_size=64, seed=SEED, penal=None):
    """Entrena `epocas` épocas con mini-batches. Devuelve el historial por época:
    loss_tr, acc_tr (entrenamiento, en modo evaluación) y loss_ev, acc_ev (conjunto de evaluación)."""
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
    criterio = nn.CrossEntropyLoss()
    hist = {"loss_tr": [], "acc_tr": [], "loss_ev": [], "acc_ev": []}
    for _ in range(epocas):
        entrenar_una_epoca(modelo, optimizador, loader, criterio, penal)
        r_tr, r_ev = evaluar(modelo, X_tr, y_tr), evaluar(modelo, X_ev, y_ev)
        hist["loss_tr"].append(r_tr["loss"]); hist["acc_tr"].append(r_tr["acc"])
        hist["loss_ev"].append(r_ev["loss"]); hist["acc_ev"].append(r_ev["acc"])
    return hist


assert sum(p.numel() for p in crear_red_base().parameters()) == 6570
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| torch", torch.__version__, "| RAPIDO =", RAPIDO)
print("entrenamiento:", Xp_t.shape[0], "| validación:", Xv_t.shape[0], "| prueba (intacta):", Xte_t.shape[0])

---
# Parte A — Optimizadores a mano en el valle de las láminas (15 min)

Se usa la misma pérdida 2D de la Teoría 2, un **valle estrecho**: casi plano a lo largo de $x$ y muy empinado a lo largo de $y$,

$$f(x, y) = \tfrac12\,(0{,}05\,x^2 + 1{,}0\,y^2), \qquad \nabla f = (0{,}05\,x,\; 1{,}0\,y),$$

con inicio en $(-10;\ 1{,}5)$. Se **cuenta un paso** cada vez que se actualiza $\theta = (x, y)$, y se para cuando $\lVert\theta\rVert < 0{,}05$.

Las cuatro reglas de las láminas (con $g = \nabla f(\theta)$):

| Optimizador | Regla |
|---|---|
| **GD** | $\theta \leftarrow \theta - \eta\, g$ |
| **Momentum** | $v \leftarrow \beta v + g$; $\quad\theta \leftarrow \theta - \eta\, v$ |
| **RMSprop** | $s \leftarrow \rho s + (1-\rho)\,g^2$; $\quad\theta \leftarrow \theta - \eta\, g\,/\,(\sqrt{s} + \varepsilon)$ |
| **Adam** | $m \leftarrow \beta_1 m + (1-\beta_1)g$; $\ v \leftarrow \beta_2 v + (1-\beta_2)g^2$; $\ \hat m = m/(1-\beta_1^t)$; $\ \hat v = v/(1-\beta_2^t)$; $\ \theta \leftarrow \theta - \eta\,\hat m\,/\,(\sqrt{\hat v} + \varepsilon)$ |

**Consigna.** Implementa cada regla en NumPy en **≤ 6 líneas**. Todas reciben `(theta, g, estado, ...)` y devuelven `(theta_nuevo, estado_nuevo)`; el estado es `None` en GD, la velocidad `v` en momentum, `s` en RMSprop y la tupla `(m, v, t)` en Adam. Después reproduce con `assert` los conteos de las láminas.

In [ ]:
CURV = np.array([0.05, 1.0])        # curvatura en x (plano) y en y (empinado)
THETA0 = np.array([-10.0, 1.5])

def f(theta):
    return 0.5 * np.sum(CURV * theta ** 2)

def grad_f(theta):
    return CURV * theta

In [ ]:
def regla_gd(theta, g, estado, eta):
    """GD: theta <- theta - eta * g. No usa estado."""
    # TODO: devuelve (theta_nuevo, estado)
    ...


def regla_momentum(theta, g, v, eta, beta):
    """Momentum: v <- beta*v + g ; theta <- theta - eta*v. Devuelve (theta, v)."""
    # TODO: actualiza la velocidad y luego theta
    ...


def regla_rmsprop(theta, g, s, eta, rho, eps=1e-8):
    """RMSprop: s <- rho*s + (1-rho)*g^2 ; theta <- theta - eta*g/(sqrt(s)+eps). Devuelve (theta, s)."""
    # TODO: promedio móvil de g**2 y paso normalizado por sqrt(s)
    ...


def regla_adam(theta, g, estado, eta, b1=0.9, b2=0.999, eps=1e-8):
    """Adam con corrección de sesgo. estado = (m, v, t). Devuelve (theta, (m, v, t))."""
    m, v, t = estado
    # TODO: t += 1 ; actualizar m y v ; corregir el sesgo (m_hat, v_hat) ; paso de theta
    ...

In [ ]:
def recorrer(regla, estado0, max_pasos=2000, **hp):
    """Aplica `regla` desde THETA0 hasta ||theta|| < 0,05. Devuelve (pasos, trayectoria)."""
    theta, estado, tray = THETA0.copy(), estado0, [THETA0.copy()]
    while np.linalg.norm(theta) >= 0.05 and len(tray) <= max_pasos:
        salida = regla(theta, grad_f(theta), estado, **hp)
        assert isinstance(salida, tuple), f"{regla.__name__} todavía no está implementada (revisa el TODO)"
        theta, estado = salida
        tray.append(np.array(theta, dtype=float))
    return len(tray) - 1, np.array(tray)


Z = np.zeros(2)
pasos, tray = {}, {}
pasos["GD"],       tray["GD"]       = recorrer(regla_gd, None, eta=1.7)
pasos["Momentum"], tray["Momentum"] = recorrer(regla_momentum, Z, eta=0.3, beta=0.8)
pasos["RMSprop"],  tray["RMSprop"]  = recorrer(regla_rmsprop, Z, eta=0.4, rho=0.9)

# Autocomprobación: los conteos de las láminas
assert pasos["GD"] == 60, f"GD (η=1,7) debería tomar 60 pasos y tomó {pasos['GD']}"
assert pasos["Momentum"] == 37, f"Momentum (η=0,3, β=0,8) debería tomar 37 pasos y tomó {pasos['Momentum']}"
assert pasos["RMSprop"] == 38, f"RMSprop (η=0,4, ρ=0,9) debería tomar 38 pasos y tomó {pasos['RMSprop']}"
print("Pasos:", pasos)

**Adam** no tiene un número en las láminas. Se reportan sus pasos con $\eta \in \{0{,}1;\ 0{,}3;\ 0{,}8\}$ ($\beta_1 = 0{,}9$, $\beta_2 = 0{,}999$) y, para ver cómo depende de $\eta$, un barrido más fino.

In [ ]:
pasos_adam, tray_adam = {}, {}
for eta in [0.1, 0.3, 0.8]:
    pasos_adam[eta], tray_adam[eta] = recorrer(regla_adam, (Z, Z, 0), eta=eta)
print("Adam, pasos por η:", pasos_adam)
assert list(pasos_adam.values()) == [232, 86, 58], "Con el código de referencia Adam toma 232, 86 y 58 pasos"

barrido = {eta: recorrer(regla_adam, (Z, Z, 0), eta=eta)[0] for eta in [0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]}
print("Barrido de η en Adam (η: pasos):", barrido)

Notar que **los pasos de Adam no bajan de forma monótona al subir $\eta$**: ver el barrido de arriba (p. ej. $\eta = 0{,}2$ y $0{,}3$, o $\eta = 0{,}8$ y $1$). Con $\eta$ grande, el paso de Adam en cada coordenada es del orden de $\eta$ sin importar la pendiente, y la trayectoria empieza a oscilar alrededor del eje del valle.

In [ ]:
def panel(ax, tray, titulo, color):
    """Curvas de nivel exactas de f (elipses) y la trayectoria con un punto por paso."""
    xs, ys = np.meshgrid(np.linspace(-11, 2, 300), np.linspace(-2.2, 2.2, 300))
    ax.contour(xs, ys, 0.5 * (CURV[0] * xs ** 2 + CURV[1] * ys ** 2), levels=[0.05, 0.2, 0.5, 1, 2, 3.5], colors=GRIS, linewidths=0.8)
    ax.plot(tray[:, 0], tray[:, 1], "-o", color=color, ms=3.5, lw=0.9)
    ax.plot(*tray[0], "s", color=MARINO, ms=8, label="inicio")
    ax.plot(0, 0, "*", color=MARINO, ms=14, label="mínimo")
    ax.set_xlim(-11, 2); ax.set_ylim(-2.2, 2.2); ax.set_aspect("equal")
    ax.set_title(titulo); ax.set_xlabel("$x$"); ax.set_ylabel("$y$")


fig, ax = plt.subplots(2, 2, figsize=(13, 6.6), constrained_layout=True)
panel(ax[0, 0], tray["GD"], f"GD (η = 1,7) — {pasos['GD']} pasos", NARANJA)
panel(ax[0, 1], tray["Momentum"], f"Momentum (η = 0,3; β = 0,8) — {pasos['Momentum']} pasos", AZUL)
panel(ax[1, 0], tray["RMSprop"], f"RMSprop (η = 0,4; ρ = 0,9) — {pasos['RMSprop']} pasos", AZUL)
panel(ax[1, 1], tray_adam[0.3], f"Adam (η = 0,3) — {pasos_adam[0.3]} pasos", AZUL)
ax[0, 0].legend(loc="lower right", fontsize=8)
plt.show()

**Ahora, GD con otra tasa.** Prueba $\eta = 2{,}1$ y $\eta = 0{,}5$ (mismo código, solo cambia `eta`).

In [ ]:
pasos_21, tray_21 = recorrer(regla_gd, None, eta=2.1, max_pasos=30)
pasos_05, tray_05 = recorrer(regla_gd, None, eta=0.5)
print(f"GD η=2,1: tras {pasos_21} pasos, ||θ|| = {np.linalg.norm(tray_21[-1]):.1f}  (inicio: {np.linalg.norm(THETA0):.1f})")
print(f"GD η=0,5: {pasos_05} pasos hasta ||θ|| < 0,05")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), constrained_layout=True)
panel(ax[0], tray_21, f"GD (η = 2,1) — diverge ({pasos_21} pasos mostrados)", NARANJA)
panel(ax[1], tray_05, f"GD (η = 0,5) — {pasos_05} pasos", NARANJA)
plt.show()

**Pregunta:** Con GD, ¿qué pasa si subes $\eta$ a 2,1? ¿Y si lo bajas a 0,5? Explica con «la dirección empinada vs la plana» (en cada eje, GD multiplica la distancia al mínimo por $1-\eta\,\lambda$, con $\lambda = 1$ en $y$ y $\lambda = 0{,}05$ en $x$).

**Respuesta:** (escribe aquí)

**Pregunta:** ¿Por qué RMSprop casi no zigzaguea aunque su $\eta$ sea fijo? (Mira los dos paneles y la forma de $g/\sqrt{s}$.)

**Respuesta:** (escribe aquí)

---
# Parte B — Optimizadores en la red de dígitos (15 min)

Misma idea, ahora en una red real. Se entrena la red base **64-64-32-10** (6.570 parámetros) durante 30 épocas con la **misma inicialización y el mismo orden de mini-batches** para:

| Optimizador | Hiperparámetros |
|---|---|
| SGD | `lr=0.1` |
| SGD + momentum | `lr=0.1`, `momentum=0.9` |
| RMSprop | `lr=1e-3` |
| Adam | `lr=1e-3` |

Se grafican las cuatro curvas de pérdida de entrenamiento en un solo gráfico y se tabula la exactitud en **validación**. Luego se prueba **Adam** con `lr ∈ {1e-4, 1e-3, 1e-2, 1e-1}`.

In [ ]:
optimizadores = {
    "SGD (lr=0.1)": lambda params: torch.optim.SGD(params, lr=0.1),
    "SGD + momentum (lr=0.1, β=0.9)": lambda params: torch.optim.SGD(params, lr=0.1, momentum=0.9),
    "RMSprop (lr=1e-3)": lambda params: torch.optim.RMSprop(params, lr=1e-3),
    "Adam (lr=1e-3)": lambda params: torch.optim.Adam(params, lr=1e-3),
}

hist_opt = {}
for nombre, fabrica in optimizadores.items():
    torch.manual_seed(SEED)                      # misma inicialización para todos
    m = crear_red_base()
    hist_opt[nombre] = entrenar(m, fabrica(m.parameters()), Xp_t, yp_t, Xv_t, yv_t, epocas=EPOCAS["opt"])

for nombre, h in hist_opt.items():
    plt.plot(h["loss_tr"], label=nombre, lw=2)
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento"); plt.title("Optimizadores: misma red, misma semilla"); plt.legend(); plt.show()

tabla_opt = pd.DataFrame({n: {"pérdida entr. final": h["loss_tr"][-1], "exactitud val. final": h["acc_ev"][-1],
                             "mejor exactitud val.": max(h["acc_ev"])} for n, h in hist_opt.items()}).T
tabla_opt.round(4)

In [ ]:
lrs_adam = [1e-4, 1e-3, 1e-2, 1e-1]
hist_adam = {}
for lr in lrs_adam:
    torch.manual_seed(SEED)
    m = crear_red_base()
    hist_adam[lr] = entrenar(m, torch.optim.Adam(m.parameters(), lr=lr), Xp_t, yp_t, Xv_t, yv_t, epocas=EPOCAS["opt"])

for lr, h in hist_adam.items():
    plt.plot(h["loss_tr"], label=f"Adam lr = {lr:g}", lw=2)
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento"); plt.title("Adam: efecto de la tasa de aprendizaje"); plt.legend(); plt.show()
pd.DataFrame({f"lr={lr:g}": {"pérdida entr. final": h["loss_tr"][-1], "exactitud val. final": h["acc_ev"][-1]} for lr, h in hist_adam.items()}).T.round(4)

**Pregunta:** ¿El optimizador que baja más rápido la pérdida de entrenamiento es el que mejor valida? ¿Qué pasa con Adam y `lr = 1e-1`? Relaciónalo con la Parte A.

**Respuesta:** (escribe aquí)

---
# Parte C — Provocar sobreajuste (8 min)

Para ver el sobreajuste hay que dejarlo ocurrir: **solo 200 imágenes** de entrenamiento y una red **más ancha** (`64 → 256 → 128 → 10`, sin regularización), con Adam (`lr=1e-3`), mini-batches de 32 y hasta 200 épocas.

Antes, `crear_mlp`: una fábrica de redes que será la base de los remedios de la Parte D.

```python
crear_mlp(capas, dropout=0.0, batchnorm=False)
```

* `capas`: lista de tamaños, p. ej. `[64, 256, 128, 10]`;
* tras cada capa lineal **oculta**: (opcional) `BatchNorm1d` → `ReLU` → (opcional) `Dropout(dropout)`;
* la última capa lineal no lleva activación.

In [ ]:
def crear_mlp(capas, dropout=0.0, batchnorm=False):
    """MLP con ReLU. Orden en cada capa oculta: Linear -> (BatchNorm1d) -> ReLU -> (Dropout)."""
    mods = []
    for i in range(len(capas) - 2):
        mods.append(nn.Linear(capas[i], capas[i + 1]))
        if batchnorm:
            mods.append(nn.BatchNorm1d(capas[i + 1]))
        mods.append(nn.ReLU())
        if dropout > 0:
            mods.append(nn.Dropout(dropout))
    mods.append(nn.Linear(capas[-2], capas[-1]))
    return nn.Sequential(*mods)


# Autocomprobación de crear_mlp
assert sum(p.numel() for p in crear_mlp([64, 64, 32, 10]).parameters()) == 6570, "Sin BatchNorm ni Dropout debe coincidir con la red base (6570 parámetros)"
assert any(isinstance(m, nn.Dropout) for m in crear_mlp([64, 64, 32, 10], dropout=0.3)), "Falta Dropout"
con_bn = crear_mlp([64, 64, 32, 10], batchnorm=True)
assert any(isinstance(m, nn.BatchNorm1d) for m in con_bn), "Falta BatchNorm1d"
assert sum(p.numel() for p in con_bn.parameters()) == 6570 + 2 * (64 + 32), "BatchNorm1d agrega γ y β por neurona oculta"
assert not isinstance(crear_mlp([64, 64, 32, 10], dropout=0.3, batchnorm=True)[-1], (nn.ReLU, nn.Dropout)), "La última capa debe ser lineal"
print("crear_mlp correcta.")

In [ ]:
X_peq, _, y_peq, _ = train_test_split(X_pool, y_pool, train_size=200, stratify=y_pool, random_state=SEED)
Xs_t, ys_t = a_tensores(X_peq, y_peq)
print("Entrenamiento pequeño:", Xs_t.shape[0], "imágenes | clases:", np.bincount(y_peq))

torch.manual_seed(SEED)
m_sobre = crear_mlp([64, 256, 128, 10])
hist_sobre = entrenar(m_sobre, torch.optim.Adam(m_sobre.parameters(), lr=1e-3), Xs_t, ys_t, Xv_t, yv_t,
                      epocas=EPOCAS["sobreajuste"], batch_size=32)
ep_min = int(np.argmin(hist_sobre["loss_ev"]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist_sobre["loss_tr"], label="entrenamiento", color=AZUL, lw=2); ax[0].plot(hist_sobre["loss_ev"], label="validación", color=NARANJA, lw=2)
ax[0].axvline(ep_min, color=MARINO, ls="--", label=f"mínimo de validación (época {ep_min + 1})")
ax[0].set_yscale("log"); ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida"); ax[0].set_title("Sobreajuste: pérdida"); ax[0].legend()
ax[1].plot(hist_sobre["acc_tr"], label="entrenamiento", color=AZUL, lw=2); ax[1].plot(hist_sobre["acc_ev"], label="validación", color=NARANJA, lw=2)
ax[1].set_xlabel("época"); ax[1].set_ylabel("exactitud"); ax[1].set_title("Sobreajuste: exactitud"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Mínimo de la pérdida de validación en la época {ep_min + 1}: {hist_sobre['loss_ev'][ep_min]:.3f}")
print(f"Al final: pérdida de entrenamiento {hist_sobre['loss_tr'][-1]:.5f} | pérdida de validación {hist_sobre['loss_ev'][-1]:.3f} | exactitud entr. {hist_sobre['acc_tr'][-1]:.3f} | exactitud val. {hist_sobre['acc_ev'][-1]:.3f}")

**Pregunta:** ¿En qué época empieza a empeorar la pérdida de validación y qué hace mientras tanto la pérdida de entrenamiento? ¿Se nota igual en la exactitud?

**Respuesta:** (escribe aquí)

---
# Parte D — Remedios, uno por uno (20 min)

Partiendo de la Parte C (**mismos 200 ejemplos, misma red ancha, misma semilla**) se prueba **una intervención a la vez**, se guarda cada resultado y al final se compara todo en una tabla.

Primero el esqueleto común: `CONFIGS` guarda cada configuración como un diccionario de opciones y `entrenar_config` la entrena; `probar` entrena, evalúa en validación y registra la fila de la tabla (pérdida, exactitud y F1 macro en validación, **brecha** = exactitud de entrenamiento − exactitud de validación, y la época de parada cuando hay *early stopping*).

In [ ]:
BASE = dict(dropout=0.0, batchnorm=False, wd=0.0, l1=0.0, es=False)
CONFIGS = {"base": BASE}
MODELOS, HIST, RES = {}, {}, {}


def registrar(nombre, modelo, hist):
    """Evalúa en entrenamiento (200 imágenes) y validación, y guarda la fila de la tabla final."""
    r_tr, r_va = evaluar(modelo, Xs_t, ys_t), evaluar(modelo, Xv_t, yv_t)
    MODELOS[nombre], HIST[nombre] = modelo, hist
    RES[nombre] = {"exactitud entr.": r_tr["acc"], "exactitud val.": r_va["acc"],
                   "F1 macro val.": f1_score(y_val, r_va["pred"], average="macro"),
                   "brecha entr.-val.": r_tr["acc"] - r_va["acc"], "pérdida val.": r_va["loss"],
                   "época de parada": len(hist["loss_ev"]) if CONFIGS[nombre]["es"] else np.nan}
    r = RES[nombre]
    print(f"{nombre:>15}: val. {r['exactitud val.']:.3f} | F1 macro {r['F1 macro val.']:.3f} | brecha {r['brecha entr.-val.']:.3f} | pérdida val. {r['pérdida val.']:.3f}")


registrar("base", m_sobre, hist_sobre)      # la Parte C es la configuración base

In [ ]:
def entrenar_config(cfg, seed=SEED, X_tr=Xs_t, y_tr=ys_t, X_ev=Xv_t, y_ev=yv_t):
    """Crea y entrena la red ancha con la configuración `cfg`. Devuelve (modelo, historial, mejor_epoca)."""
    torch.manual_seed(seed)
    m = crear_mlp([64, 256, 128, 10], dropout=cfg["dropout"], batchnorm=cfg["batchnorm"])
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=cfg["wd"])
    penal = (lambda mod: penalizacion_l1(mod, cfg["l1"])) if cfg["l1"] > 0 else None   # penalizacion_l1: D2
    if cfg["es"]:
        hist, mejor = entrenar_early_stopping(m, opt, X_tr, y_tr, X_ev, y_ev, epocas_max=EPOCAS["sobreajuste"], paciencia=10, seed=seed, penal=penal)   # entrenar_early_stopping: D5
    else:
        hist = entrenar(m, opt, X_tr, y_tr, X_ev, y_ev, epocas=EPOCAS["sobreajuste"], batch_size=32, seed=seed, penal=penal)
        mejor = None
    return m, hist, mejor


def probar(nombre):
    m, h, _ = entrenar_config(CONFIGS[nombre])
    registrar(nombre, m, h)

## D1. L2 (*weight decay*)

`weight_decay=1e-3` en Adam: suma $\tfrac{\lambda}{2}\lVert w\rVert^2$ a la pérdida (su gradiente es $\lambda w$) y empuja los pesos hacia cero.
*Nota:* con Adam también existe `AdamW`, que aplica el decaimiento por separado del paso adaptativo; aquí no se usa.

In [ ]:
CONFIGS["L2"] = {**BASE, "wd": 1e-3}
probar("L2")

## D2. L1

L1 suma $\lambda \sum |w|$ a la pérdida, con $\lambda = 10^{-3}$, **solo sobre los pesos** de las capas lineales (no los sesgos). A diferencia de L2, su gradiente no se achica cerca de cero ($\lambda\,\mathrm{signo}(w)$), así que empuja muchos pesos hasta (casi) cero: promueve **dispersión**.

**Consigna.** Completa `penalizacion_l1(modelo, lam)`. `entrenar` ya la suma a la pérdida de cada lote (parámetro `penal`).

In [ ]:
def penalizacion_l1(modelo, lam):
    """lam * suma de |w| sobre los pesos de las capas nn.Linear (sin sesgos)."""
    # TODO: recorre modelo.modules(), toma las capas nn.Linear y suma .weight.abs().sum(); multiplica por lam
    return ...


# Autocomprobación: 0,1 * (1+2+3+4) = 1,0 (los sesgos no cuentan)
_l = nn.Linear(2, 2)
with torch.no_grad():
    _l.weight.copy_(torch.tensor([[1., -2.], [3., -4.]])); _l.bias.copy_(torch.tensor([10., 10.]))
_p = penalizacion_l1(nn.Sequential(_l), 0.1)
assert isinstance(_p, torch.Tensor) and _p.requires_grad, "Debe devolver un tensor diferenciable (para que backward() lo incluya)"
assert abs(_p.item() - 1.0) < 1e-6, f"Esperaba 0,1·(1+2+3+4) = 1,0 y obtuve {_p.item():.4f}: ¿incluiste los sesgos o olvidaste el valor absoluto?"
print("penalizacion_l1 correcta.")

Se entrena L1 (tarda un poco más: calcula la penalización en cada lote) y, junto con L2, se compara qué tan dispersos quedan los pesos: fracción de pesos con $|w| < 10^{-3}$ en la **base**, **L2** y **L1**, y el histograma de $|w|$ de la primera capa en escala log.

In [ ]:
CONFIGS["L1"] = {**BASE, "l1": 1e-3}
probar("L1")


def frac_ceros(m, umbral=1e-3):
    """Fracción de pesos (de las capas nn.Linear) con |w| < umbral."""
    w = torch.cat([l.weight.detach().abs().flatten() for l in m.modules() if isinstance(l, nn.Linear)])
    return (w < umbral).float().mean().item()


fr = {n: frac_ceros(MODELOS[n]) for n in ["base", "L2", "L1"]}
print("Fracción de pesos con |w| < 1e-3 —", " | ".join(f"{n}: {v:.1%}" for n, v in fr.items()))

colores = {"base": MARINO, "L2": AZUL, "L1": NARANJA}
bins = np.logspace(-7, 0.5, 60)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
for n, c in colores.items():
    w = MODELOS[n][0].weight.detach().abs().flatten().numpy()
    ax[0].hist(np.clip(w, 1e-7, None), bins=bins, histtype="step", lw=2, color=c, label=f"{n} (|w| < 1e-3: {np.mean(w < 1e-3):.0%})")
ax[0].axvline(1e-3, color=GRIS, ls=":"); ax[0].set_xscale("log")
ax[0].set_xlabel("|w| de la primera capa (escala log)"); ax[0].set_ylabel("n.º de pesos"); ax[0].set_title("Primera capa: L2 achica, L1 anula"); ax[0].legend(fontsize=9)
ax[1].bar(list(fr), [100 * v for v in fr.values()], color=list(colores.values()))
ax[1].set_ylabel("% de pesos con |w| < 1e-3 (toda la red)"); ax[1].set_title("Dispersión")
plt.tight_layout(); plt.show()

## D3. Batch Normalization

Primero **el ejemplo de la lámina**. `nn.BatchNorm1d(1)` en modo `train()` normaliza el lote con su media y su varianza (la **poblacional**, $\sigma^2 = 5$ para $[2, 4, 6, 8]$) y luego aplica $\gamma\,\hat x + \beta$.

In [ ]:
x = torch.tensor([[2.], [4.], [6.], [8.]])

bn = nn.BatchNorm1d(1)                       # γ = 1, β = 0
bn.train()
y1 = bn(x).detach().flatten()
assert torch.allclose(y1, torch.tensor([-1.34, -0.45, 0.45, 1.34]), atol=1e-2), y1
print("γ=1, β=0 (train):", y1.numpy().round(2))

bn2 = nn.BatchNorm1d(1)
bn2.weight.data.fill_(2); bn2.bias.data.fill_(1)      # γ = 2, β = 1
bn2.train()
y2 = bn2(x).detach().flatten()
assert torch.allclose(y2, torch.tensor([-1.68, 0.11, 1.89, 3.68]), atol=1e-2), y2
print("γ=2, β=1 (train):", y2.numpy().round(2))

print("running_mean =", bn2.running_mean.item(), "| running_var =", round(bn2.running_var.item(), 4))   # tras UNA pasada

Los promedios móviles se actualizan con `momentum = 0,1`: $\text{running} \leftarrow 0{,}9\cdot\text{running} + 0{,}1\cdot\text{valor del lote}$. Parten de media 0 y varianza 1, así que tras una pasada `running_mean` $= 0{,}9\cdot 0 + 0{,}1\cdot 5 = 0{,}5$ y `running_var` $= 0{,}9\cdot 1 + 0{,}1\cdot 6{,}67 = 1{,}57$. Ojo: `running_var` usa la varianza **muestral** (6,67, dividiendo por $n-1$), por eso no es 5.

En `eval()` la capa **no mira el lote**: usa esos promedios móviles. Mira cómo cambia la salida.

In [ ]:
bn2.eval()
print("γ=2, β=1 (eval, tras 1 pasada):", bn2(x).detach().flatten().numpy().round(2), " <- usa running_mean=0,5 y running_var=1,57")

bn2.train()
for _ in range(100):                          # 100 pasadas más: los promedios móviles convergen a los del lote
    bn2(x)
bn2.eval()
print("running_mean =", round(bn2.running_mean.item(), 3), "| running_var =", round(bn2.running_var.item(), 3))
print("γ=2, β=1 (eval, tras 100 pasadas):", bn2(x).detach().flatten().numpy().round(2), " (≠ de train porque usa 6,67 y no 5)")

**En la red.** `BatchNorm1d` después de cada lineal oculta y **antes** de la ReLU (el esquema de la lámina: Lineal → BatchNorm → ReLU). `crear_mlp(..., batchnorm=True)` ya lo hace; `evaluar` pone el modelo en `eval()` para usar los promedios móviles.

In [ ]:
CONFIGS["BatchNorm"] = {**BASE, "batchnorm": True}
print(crear_mlp([64, 256, 128, 10], batchnorm=True)[:3])
probar("BatchNorm")

## D4. Dropout

`nn.Dropout(0.3)` después de cada ReLU oculta: en cada paso apaga al azar el 30 % de las neuronas (y escala el resto por $1/0{,}7$). Solo actúa en `model.train()`; en `model.eval()` no hace nada. Por eso `entrenar_una_epoca` llama a `modelo.train()` y `evaluar` a `modelo.eval()`.

In [ ]:
dp = nn.Dropout(0.3)
u = torch.ones(1, 10)
dp.train(); print("train:", dp(u).numpy().round(2))
dp.eval();  print("eval: ", dp(u).numpy().round(2))

CONFIGS["Dropout"] = {**BASE, "dropout": 0.3}
probar("Dropout")

## D5. Early stopping

Se entrena hasta 200 épocas, se mide la **pérdida de validación** al final de cada una y, si no mejora durante `paciencia = 10` épocas seguidas, se detiene y se **restauran los pesos de la mejor época**.

**Consigna.** Completa el ciclo de `entrenar_early_stopping` y comprueba con la autocomprobación de la celda siguiente.

In [ ]:
def entrenar_early_stopping(modelo, optimizador, X_tr, y_tr, X_ev, y_ev, epocas_max=200, paciencia=10, batch_size=32, seed=SEED, penal=None):
    """Early stopping sobre la pérdida de validación. Restaura los mejores pesos.
    Devuelve (historial, mejor_epoca) con mejor_epoca contada desde 1."""
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
    criterio = nn.CrossEntropyLoss()
    hist = {"loss_tr": [], "acc_tr": [], "loss_ev": [], "acc_ev": []}
    mejor_loss, mejor_epoca, mejores_pesos, sin_mejora = float("inf"), 0, None, 0
    for ep in range(1, epocas_max + 1):
        entrenar_una_epoca(modelo, optimizador, loader, criterio, penal)
        r_tr, r_ev = evaluar(modelo, X_tr, y_tr), evaluar(modelo, X_ev, y_ev)
        hist["loss_tr"].append(r_tr["loss"]); hist["acc_tr"].append(r_tr["acc"])
        hist["loss_ev"].append(r_ev["loss"]); hist["acc_ev"].append(r_ev["acc"])
        # TODO: si la pérdida de validación (r_ev["loss"]) mejoró -> guardar una COPIA de modelo.state_dict() (copy.deepcopy),
        #       actualizar mejor_loss / mejor_epoca y reiniciar el contador `sin_mejora`;
        #       si no mejoró, sumar 1 al contador y salir del ciclo (break) cuando llegue a `paciencia`
        ...
    # TODO: restaurar los mejores pesos con modelo.load_state_dict
    ...
    return hist, mejor_epoca

In [ ]:
# Autocomprobación del early stopping
torch.manual_seed(SEED)
_m = crear_mlp([64, 256, 128, 10])
_h, _ep = entrenar_early_stopping(_m, torch.optim.Adam(_m.parameters(), lr=1e-3), Xs_t, ys_t, Xv_t, yv_t, epocas_max=EPOCAS["sobreajuste"])
assert len(_h["loss_ev"]) < EPOCAS["sobreajuste"], "El early stopping debería detener antes de las épocas máximas en este problema"
assert len(_h["loss_ev"]) == _ep + 10, "Debe detenerse exactamente 10 épocas después de la mejor (paciencia = 10)"
assert abs(evaluar(_m, Xv_t, yv_t)["loss"] - min(_h["loss_ev"])) < 1e-5, "Tras el entrenamiento el modelo debe tener los pesos de la MEJOR época"
print(f"Early stopping: se detuvo en la época {len(_h['loss_ev'])}; mejores pesos de la época {_ep}.")

CONFIGS["Early stopping"] = {**BASE, "es": True}
probar("Early stopping")
h = HIST["Early stopping"]
plt.plot(h["loss_tr"], label="entrenamiento", color=AZUL, lw=2); plt.plot(h["loss_ev"], label="validación", color=NARANJA, lw=2)
plt.axvline(_ep - 1, color=MARINO, ls="--", label=f"mejores pesos (época {_ep})")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida"); plt.title(f"Early stopping: se detiene en la época {len(h['loss_ev'])}"); plt.legend(); plt.show()

## D6. Combinación

Se combinan **las tres intervenciones que mejoraron la validación** en las celdas anteriores: **BatchNorm + Dropout (0,3) + early stopping**. L2 y L1 (con $\lambda = 10^{-3}$) no mejoraron la exactitud de validación, así que se dejan fuera.

In [ ]:
CONFIGS["Combinación"] = {**BASE, "batchnorm": True, "dropout": 0.3, "es": True}
probar("Combinación")

### Tabla final

Configuración, exactitud y F1 macro en validación, brecha entre entrenamiento y validación, y época de parada cuando aplica.

In [ ]:
tabla = pd.DataFrame(RES).T.loc[list(CONFIGS)]
display(tabla.round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
PALETA = [GRIS, AZUL, NARANJA, MARINO, "#5a8a6a", "#c9a27a", "#9aa6b8", "#6c7480"]
for i, (nombre, h) in enumerate(HIST.items()):
    ax[0].plot(h["loss_ev"], label=nombre, lw=1.8, color=PALETA[i % 8])
    ax[1].plot(h["acc_ev"], label=nombre, lw=1.8, color=PALETA[i % 8])
ax[0].set_yscale("log"); ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida de validación"); ax[0].set_title("Pérdida de validación por configuración"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("época"); ax[1].set_ylabel("exactitud de validación"); ax[1].set_title("Exactitud de validación por configuración"); ax[1].set_ylim(0.8, 1.0)
plt.tight_layout(); plt.show()

**Pregunta:** Según la tabla, ¿qué intervención redujo más la brecha entre entrenamiento y validación y cuál mejoró más la validación? ¿Ayuda la combinación? Explica **en una frase por técnica** por qué actúa como actúa (L2, L1, BatchNorm, Dropout, early stopping).

**Respuesta:** (escribe aquí)

---
# Parte E — Informe con incertidumbre (7 min)

Se elige la mejor configuración **según la validación** (F1 macro) y recién entonces se mira la **prueba, una sola vez** (la receta de la clase 9).

1. Reentrena esa configuración con **3 semillas** (`SEED`, `SEED+1`, `SEED+2`) y reporta media ± desviación estándar de la exactitud de prueba.
2. Para el modelo de la primera semilla, calcula el **IC95 % por bootstrap** de la exactitud de prueba ($B = 1000$ remuestreos con reemplazo, percentiles 2,5 y 97,5).
3. Escribe tres conclusiones.

In [ ]:
def ic_bootstrap_exactitud(y_real, y_pred, B=1000, nivel=0.95, seed=SEED):
    """IC percentil por bootstrap de la exactitud. Devuelve ((inferior, superior), distribución_bootstrap)."""
    rng = np.random.default_rng(seed)
    aciertos = (np.asarray(y_real) == np.asarray(y_pred)).astype(float)
    n = len(aciertos)
    medias = np.array([aciertos[rng.integers(0, n, n)].mean() for _ in range(B)])
    alfa = (1 - nivel) / 2
    return tuple(np.quantile(medias, [alfa, 1 - alfa])), medias


# Autocomprobación con un caso conocido: 90 aciertos de 100
(lo, hi), dist = ic_bootstrap_exactitud(np.r_[np.ones(90), np.zeros(10)], np.ones(100))
assert len(dist) == 1000 and 0.82 < lo < 0.90 < hi < 0.97, f"IC inesperado para 90/100: ({lo:.3f}, {hi:.3f})"
print(f"IC95 % de 90/100 aciertos: [{lo:.3f}, {hi:.3f}]  (correcto)")

In [ ]:
# TODO: elige la configuración con mayor F1 macro de validación (una columna de `tabla`; el índice de `tabla` son los nombres de CONFIGS)
mejor = ...

assert mejor in CONFIGS, "`mejor` debe ser el nombre de una configuración (p. ej. 'BatchNorm')"
print("Mejor configuración según F1 macro de validación:", mejor, "->", CONFIGS[mejor])

In [ ]:
exact_test, preds_test = [], []
for k in range(3):
    m, _, _ = entrenar_config(CONFIGS[mejor], seed=SEED + k)
    r = evaluar(m, Xte_t, yte_t)          # <- la prueba se usa UNA vez por modelo final
    exact_test.append(r["acc"]); preds_test.append(r["pred"])

exact_test = np.array(exact_test)
(lo, hi), dist = ic_bootstrap_exactitud(y_te, preds_test[0])
print(f"Exactitud de prueba por semilla: {np.round(exact_test, 4)}")
print(f"Media ± desviación estándar (3 semillas): {exact_test.mean():.4f} ± {exact_test.std(ddof=1):.4f}")
print(f"IC95 % bootstrap (semilla {SEED}, n = {len(y_te)}): [{lo:.4f}, {hi:.4f}]")

plt.hist(dist, bins=30, color="#9aa6b8", edgecolor="white")
plt.axvline(lo, color=NARANJA, ls="--"); plt.axvline(hi, color=NARANJA, ls="--")
plt.axvline(exact_test[0], color=MARINO); plt.xlabel("exactitud en prueba (remuestreo)"); plt.ylabel("frecuencia")
plt.title("Distribución bootstrap de la exactitud (líneas naranja: IC95 %)"); plt.show()

### Conclusiones

Escribe tres conclusiones: qué optimizador, qué remedio funcionó mejor y por qué, y qué tan seguro es el resultado final.

**Conclusión 1** — Optimizador:

**Respuesta:** (escribe aquí)

**Conclusión 2** — Remedio contra el sobreajuste:

**Respuesta:** (escribe aquí)

**Conclusión 3** — Incertidumbre del resultado final:

**Respuesta:** (escribe aquí)

---
## Entregable

`Lab10b_Entrenar_estudiante.ipynb` **ejecutado**, con las preguntas respondidas y las **tres conclusiones**. Entrega de la clase: los dos notebooks, `Lab10a` y `Lab10b`, por AULA-USM.

In [ ]:
print(f"Tiempo total de ejecución: {time.time() - T0:.0f} s")